In [2]:
import sys
sys.path.append('../')

import torch
from transformers import RobertaTokenizerFast
from hf_pretrained_model import RobertaConfigHF, RobertaForQAHF
from transformers import AutoModel, AutoConfig

import warnings
warnings.filterwarnings('ignore')

In [3]:
AutoConfig.register('roberta-qa', RobertaConfigHF)
AutoModel.register(RobertaConfigHF, RobertaForQAHF)
config = AutoConfig.from_pretrained('detker/roberta-qa-125M')
model = AutoModel.from_pretrained('detker/roberta-qa-125M',
                                  trust_remote_code=True,
                                  use_safetensors=True,
                                  config=config)
tokenizer = RobertaTokenizerFast.from_pretrained(config.hf_model_name)
model.eval()

Some weights of RobertaModel were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForQAHF(
  (model): RobertaForQA(
    (model): RobertaModel(
      (embeddings): RobertaEmbeddings(
        (word_embeddings): Embedding(50265, 768, padding_idx=1)
        (position_embeddings): Embedding(514, 768, padding_idx=1)
        (token_type_embeddings): Embedding(1, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): RobertaEncoder(
        (layer): ModuleList(
          (0-11): 12 x RobertaLayer(
            (attention): RobertaAttention(
              (self): RobertaSdpaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): RobertaSelfOutput(
                (dense): Linear(in_feature

In [4]:
def inference(question, context):
    inputs = tokenizer(
        text=question,
        text_pair=context,
        max_length=config.context_length,
        truncation='only_second',
        return_tensors='pt'
    )

    with torch.no_grad():
        start_logits, end_logits = model(inputs)

    start_token_idx = start_logits.squeeze().argmax().item()
    end_token_idx = end_logits.squeeze().argmax().item()

    tokens = inputs['input_ids'].squeeze()[start_token_idx:end_token_idx+1]
    answer = tokenizer.decode(tokens, skip_special_tokens=True).strip()

    return start_token_idx, end_token_idx, answer

In [5]:
context = "Water boils at 100 degrees Celsius at sea level, but the boiling point decreases at higher altitudes."
question = "At what temperature does water boil at sea level?"

context2 = """
        The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. 
        It is named after the engineer Gustave Eiffel, whose company designed and built the tower. 
        Constructed from 1887 to 1889 as the entrance to the 1889 World's Fair, it was initially criticized 
        by some of France's leading artists and intellectuals for its design, but it has become a global 
        cultural icon of France."""
question2 = "When was the Eiffel Tower constructed?"

context3 = """
        Neural networks are a subset of machine learning and are at the heart of deep learning algorithms. 
        Their name and structure are inspired by the human brain, mimicking the way that biological neurons 
        signal to one another. A typical neural network is composed of layers of nodes, much like the neurons 
        of the human brain."""
question3 = "What inspired the structure of neural networks?"

start_token_idx, end_token_idx, answer = inference(question, context)
print(f'{start_token_idx}-{end_token_idx}: {answer}')
start_token_idx, end_token_idx, answer = inference(question2, context2)
print(f'{start_token_idx}-{end_token_idx}: {answer}')
start_token_idx, end_token_idx, answer = inference(question3, context3)
print(f'{start_token_idx}-{end_token_idx}: {answer}')

16-18: 100 degrees Celsius
84-87: 1887 to 1889
54-56: human brain,
